# eDAS .bz Reader Notebook

Read packetized Bitshuffle + Zstd `.bz` files produced by the GUI, print data volume, plot one point waveform, and draw a time-space image.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'bz_format.py').exists():
            return candidate
    fallback = Path(r'E:/codes/PCIe-7821/pcie7821_gui')
    if (fallback / 'src' / 'bz_format.py').exists():
        return fallback
    raise FileNotFoundError('Cannot locate repo root containing src/bz_format.py')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / 'src'))
from bz_format import iter_bz_packets

print('repo:', REPO_ROOT)

In [ ]:
# Change these values before running the following cells.
BZ_PATH = Path(r'D:/eDAS_DATA/example.bz')
VERIFY_CRC = True
MAX_PACKETS = None  # Use an int such as 10 for a quick preview.

POINT_INDEX = 0
CHANNEL_INDEX = 0

SPACE_START = 0
SPACE_END = None
FRAME_START = 0
FRAME_COUNT = None
TIME_DOWNSAMPLE = 1
SPACE_DOWNSAMPLE = 1

In [ ]:
if not BZ_PATH.exists():
    raise FileNotFoundError(BZ_PATH)

packet_arrays = []
packet_headers = []
file_info = None
for idx, (fi, pi, samples) in enumerate(iter_bz_packets(BZ_PATH, verify_crc=VERIFY_CRC)):
    if file_info is None:
        file_info = fi
    packet_headers.append(pi)
    packet_arrays.append(samples)
    if MAX_PACKETS is not None and idx + 1 >= int(MAX_PACKETS):
        break

if not packet_arrays:
    raise ValueError('No packet found in .bz file')

packet_matrix = np.concatenate(packet_arrays, axis=0)
scan_rate_hz = int(file_info.get('scan_rate_hz', packet_headers[0]['scan_rate_hz']))
points_per_frame = int(file_info.get('points_per_frame', packet_headers[0]['points_per_frame']))
channel_num = int(file_info.get('channel_num', 1))
packet_width = int(packet_headers[0]['points_per_frame'])

if channel_num > 1:
    expected_width = points_per_frame * channel_num
    if packet_width != expected_width:
        raise ValueError(f'packet width {packet_width} != points_per_frame*channel_num {expected_width}')
    data = packet_matrix.reshape(packet_matrix.shape[0], points_per_frame, channel_num)
else:
    data = packet_matrix.reshape(packet_matrix.shape[0], points_per_frame)

total_frames = int(data.shape[0])
total_values = int(data.size)
raw_bytes = total_values * np.dtype(np.int32).itemsize
duration_s = total_frames / float(scan_rate_hz)
compressed_file_bytes = BZ_PATH.stat().st_size

print('file:', BZ_PATH)
print('file_info:', file_info)
print('packets:', len(packet_headers))
print('data_shape:', data.shape)
print('scan_rate_hz:', scan_rate_hz)
print('points_per_frame:', points_per_frame)
print('channel_num:', channel_num)
print('frames:', total_frames)
print('duration_s:', duration_s)
print('int32_values:', total_values)
print('raw_bytes:', raw_bytes)
print('compressed_file_bytes:', compressed_file_bytes)
print('raw/compressed ratio:', raw_bytes / compressed_file_bytes if compressed_file_bytes else np.inf)

In [ ]:
point = int(np.clip(POINT_INDEX, 0, points_per_frame - 1))
channel = int(np.clip(CHANNEL_INDEX, 0, max(1, channel_num) - 1))
time_axis = np.arange(total_frames, dtype=float) / float(scan_rate_hz)
waveform = data[:, point, channel] if channel_num > 1 else data[:, point]

plt.figure(figsize=(12, 4))
plt.plot(time_axis, waveform, linewidth=0.8)
plt.xlabel('Time (s)')
plt.ylabel('int32 value')
plt.title(f'Point {point}, Channel {channel}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
frame_start = max(0, int(FRAME_START))
frame_end = total_frames if FRAME_COUNT is None else min(total_frames, frame_start + int(FRAME_COUNT))
space_start = max(0, int(SPACE_START))
space_end = points_per_frame if SPACE_END is None else min(points_per_frame, int(SPACE_END))
tds = max(1, int(TIME_DOWNSAMPLE))
sds = max(1, int(SPACE_DOWNSAMPLE))
channel = int(np.clip(CHANNEL_INDEX, 0, max(1, channel_num) - 1))

block = data[frame_start:frame_end:tds, space_start:space_end:sds, channel] if channel_num > 1 else data[frame_start:frame_end:tds, space_start:space_end:sds]
extent = [frame_start / scan_rate_hz, max(frame_start, frame_end - 1) / scan_rate_hz, space_start, max(space_start, space_end - 1)]

plt.figure(figsize=(12, 6))
plt.imshow(block.T, aspect='auto', origin='lower', extent=extent, interpolation='nearest')
plt.xlabel('Time (s)')
plt.ylabel('Point index')
plt.title(f'Time-Space, Channel {channel}, shape={block.shape}')
plt.colorbar(label='int32 value')
plt.tight_layout()
plt.show()

In [ ]:
fields = ['packet_index', 'frames', 'points_per_frame', 'raw_bytes', 'compressed_bytes', 'zstd_level', 'bitshuffle_block_values']
for header in packet_headers[:20]:
    print({field: header[field] for field in fields})